# CERN Image Archive Webscraper

In [53]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.common.keys import Keys  # Add this line
from selenium.webdriver.support import expected_conditions as EC
import re
import time
import os
import logging
from datetime import datetime
from bs4 import BeautifulSoup
from bs4 import NavigableString
from urllib.parse import urljoin
import csv
import pandas as pd

### Global Variables

Since the first page is different than any other we start the code from page 2. 
The content of page one can be collected by hand (10 records) which is probably faster than coding the special case. 

In [66]:
# Global Variables: Change when you on each scrapping session
JREC_START = 131 # new number = last JREC_END + 1
PAGES_TO_SCRAPE = 3 # how many pages? controls the loop

# each page holds 10 records, so the last jrec of this session is PAGES_TO_SCRAPE * 10 + The Starting Point+1
JREC_END = PAGES_TO_SCRAPE * 10 + (JREC_START - 1)

print("Start: ", JREC_START, " End: ", JREC_END)
print(PAGES_TO_SCRAPE, "Pages to scrape")

# create a csv file for each scraping session with the record start/end in the filename
# you can use this to see were you need to continue
EXPORT_CSV_PATH = f'./export-scrape/cern-photoarchive-page-{JREC_START}-{JREC_END}.csv'

# base url for reference
BASE_SEARCH_URL = "https://cds.cern.ch/collection/PhotoLab%20Archives?ln=en"

Start:  131  End:  160
3 Pages to scrape


### 1. Setup

The setup does 3 things: 
1. Open the page
2. Jump to page two, as from there on all pages have the same structure
3. Jump to the page you want to start scraping by entering the record start number

In [67]:
# 1. Open Page
# -------------------------

# Start driver
driver = webdriver.Firefox()

# Open list page
driver.get(BASE_SEARCH_URL)

# Give page time to load
time.sleep(2)

# 2. Jump to page twi
# -------------------------

# Click "[>> more]"
more_link = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.LINK_TEXT, "[>> more]"))
)
more_link.click()
time.sleep(2)

# 3. Jump to the page you want to start
# -------------------------

jrec_input = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='jrec']"))
)

# set the new value based on the above variable
jrec_input.clear()
jrec_input.send_keys(str(JREC_START))
jrec_input.send_keys(Keys.RETURN)

print("Ready to (continoute to) scrape!")

Ready to (continoute to) scrape!


### 3. Start scraping

1. First we create a function that we can reuse to scrape the actual content from the page
2. Then we build a loop to scrape as many pages as defined and handle the navigation to these pages

In [68]:
titles = []
links = []
record_urls = []
image_links = []
image_filenames = []
dates = []
tirages = []
descriptions = []

# -------------------------
# Scraping Function
# -------------------------

def scrape_current_page(page_idx, current_jrec):    
    soup = BeautifulSoup(driver.page_source, "html.parser")

    results = [
        tr for tr in soup.select("tbody > tr")
        if len(tr.find_all("td")) == 2
    ]

    print(f"Page {page_idx+1}/{PAGES_TO_SCRAPE} (jrec={current_jrec}) - Found rows:", len(results))

    # -------------------------
    # DATES
    # -------------------------

    date_pattern = re.compile(
        r"(?i)(\d{1,2}\s+[A-Za-z]{3,9}\s+\d{4})|"
        r"([A-Za-z]{3,9}\s+\d{4})|"
        r"(\d{4})|"
        r"(No date)"
    )

    for r in results:
        td = r.find_all("td")[1]
        date_text = None

        for br in td.find_all("br"):
            sib = br.next_sibling
            while sib and (not isinstance(sib, NavigableString) or not sib.strip()):
                sib = sib.next_sibling
            if not sib:
                continue

            text = sib.strip()
            if "[...]" in text:
                continue

            m = date_pattern.search(text)
            if m:
                date_text = m.group(0)
                break

        dates.append(date_text)

    # -------------------------
    # TIRAGES
    # -------------------------
    for r in results:
        td = r.find_all("td")[1]
        tirage_text = None

        em = td.find("em", string=lambda s: s and "Tirage" in s)
        if em:
            sib = em.next_sibling
            while sib and (not isinstance(sib, NavigableString) or not sib.strip()):
                sib = sib.next_sibling
            if sib:
                tirage_text = sib.strip()
        # get rif of ': '
        tirages.append(tirage_text[2:])
        
    # -------------------------
    # TITLES + LINKS + IMAGES
    # -------------------------
    base_url = "https://cds.cern.ch"

    for r in results:
        cells = r.find_all("td")

        a = cells[1].find("a", class_="titlelink")
        if a:
            titles.append(a.get_text(strip=True))
            links.append(a.get("href"))
            record_urls.append(urljoin(base_url, a.get("href")))
        else:
            titles.append(None)
            links.append(None)
            record_urls.append(None)

        img = cells[0].find("img")
        if img and img.get("src"):
            img_url = urljoin(base_url, img.get("src"))
            image_links.append(img_url)

            if "/files/" in img_url:
                fname = img_url.split("/files/")[1].split(".jpg")[0]
            else:
                fname = None

            image_filenames.append(fname)
        else:
            image_links.append(None)
            image_filenames.append(None)

    # -------------------------
    # DESCRIPTIONS: DOES NOT WORK, PRODUCES NONE
    # -------------------------
    # for record_url in record_urls:
    #     if not record_url:
    #         descriptions.append(None)
    #         continue

    #     driver.get(record_url)
    #     time.sleep(1)

    #     soup = BeautifulSoup(driver.page_source, "html.parser")
    #     desc_divs = soup.select("div.album-description")

    #     parts = []
    #     for div in desc_divs:
    #         for p in div.find_all("p"):
    #             t = p.get_text(strip=True)
    #             if t:
    #                 parts.append(t)

    #     descriptions.append(" ".join(parts) if parts else None)

In [69]:
# loop logic calls the above function & handles navigation

# ATTENTION: If you run this again it will add the same results again to the above devined lists
# this leads to duplicates, therefore we "empty" the lists first
titles = []
links = []
record_urls = []
image_links = []
image_filenames = []
dates = []
tirages = []
descriptions = []

for page_idx in range(PAGES_TO_SCRAPE):

    # calculate JREC position for current loop
    current_jrec = JREC_START + page_idx * 10
    
    # Jump to correct jrec (also works for first page)
    jrec_input = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='jrec']")))
    jrec_input.clear()
    jrec_input.send_keys(str(current_jrec))
    jrec_input.send_keys(Keys.RETURN)

    # Wait until navigation/update happened (URL often includes jrec=...)
    WebDriverWait(driver, 10).until(EC.url_contains(f"jrec={current_jrec}"))

    # Optional small sleep if site is slow to render table after URL update
    time.sleep(0.5)
    
    # now we are on the page and can call the function that collects all the data
    scrape_current_page(page_idx, current_jrec)

Page 1/3 (jrec=131) - Found rows: 10
Page 2/3 (jrec=141) - Found rows: 10
Page 3/3 (jrec=151) - Found rows: 10


3. Checking the Results

In [70]:
# lets see what we found
# length should be Scapes * 10
print("Found dates:", len(dates), dates)
print("Found tirages:", len(tirages), tirages)
# NOTE: links and record_urls seem to be the same
print("Found titles:", len(titles), titles)
print("Found links:", len(links), links)
print("Found record_urls:", len(record_urls), record_urls)
print("Found image_filenames:", len(image_filenames), image_filenames)
print("Found image_links:", len(image_links), image_links)

Found dates: 30 ['Mar 1978', 'Feb 1975', 'Feb 1975', 'Feb 1975', 'Feb 1975', 'Feb 1975', 'Mar 1971', 'Jun 1972', 'Jan 1974', 'Oct 1971', 'Nov 1971', '8 Apr 1975', 'Feb 1972', '24 Aug 1972', '15 Jun 1972', 'Sep 1971', 'Feb 1975', 'Feb 1975', 'Feb 1975', 'Feb 1975', 'Jun 1972', '5 Feb 1975', 'Apr 1973', 'Jan 1975', 'Jan 1975', 'Jan 1975', 'Nov 1974', 'Oct 1974', 'Oct 1974', 'Oct 1974']
Found tirages: 30 ['4', '1', '1', '2', '6', '3', '2', '8', '3', '0', '3', '4', '1', '3', '3', '16', '4', '2', '3', '2', '1', '1', '2', '2', '2', '1', '4', '1', '1', '2']
Found titles: 30 ['Universal testing machine', 'Album title to be added', 'Vacuum equipment in a hall', 'Electrically controlled angle valves', 'Apparatus for orbital electrical arc welding under protective atmosphere (argon) from ESAB', 'Album title to be added', 'Vacuum control racks at the ISR', 'Water cooling towers and water treatment station, Building 176', 'Album title to be added', 'Rotary capacitor', 'Damaged component', 'Work ins

### 3. Export to CSV

In [71]:
# -------------------------
# BUILD DF + EXPORT CSV
# -------------------------
df = pd.DataFrame({
    "title": titles,
    "link": links,
    "record_url": record_urls,
    "image_link": image_links,
    "image_filename": image_filenames,
    "date": dates,
    "tirage": tirages,
    # "description": descriptions,  # enable when you actually fill it
})

df.to_csv(EXPORT_CSV_PATH, index=False)
print("Saved:", EXPORT_CSV_PATH)

driver.quit()

Saved: ./export-scrape/cern-photoarchive-page-131-160.csv
